# TR-MMLU Benchmark — Yalın Qwen3-14B vs MEB LoRA

**Amaç:** `namruni/meb-ogretmen-qwen3-14b-lora` fine-tune modelini, yalın taban model
`unsloth/Qwen3-14B` ile TR-MMLU (`alibayram/turkish_mmlu` → `mmlu` split, 6200 soru)
üzerinde **aynı koşullarda** kıyaslamak. Kontrollü deney: tek değişen = LoRA yaması.

### Çalıştırmadan önce (ön koşullar)
1. **Runtime → Change runtime type → GPU** (L4 ya da A100) seç.
2. Veri seti *gated*: <https://huggingface.co/datasets/alibayram/turkish_mmlu> sayfasına
   git, **şartları bir kez kabul et** (Agree). Aksi halde indirmek 401 verir.
3. Bir **HF token'ın** hazır olsun (huggingface.co → Settings → Access Tokens, `read` yetkili).
   Token'ı koda YAZMIYORUZ; Hücre 4'te güvenli biçimde soracağız.

### Nasıl ilerleyelim
Hücreleri **sırayla** çalıştır. Bir hücre hata verirse, **hücre numarasını** söyle;
o hücrenin kodunu düzeltirim. Önce **pilot (~100 soru)**, sonra tam sete büyütürüz.

> Atıf: Bayram, M. Ali ve diğerleri — TR-MMLU (Zenodo DOI ile atıf yapılmalı).


In [ ]:
# ===== HÜCRE 2: Kütüphane kurulumu =====
# unsloth'u kurar; uyumlu torch/transformers/peft sürümlerini de kendisi getirir.
# Not: "hash mismatch / DO NOT MATCH THE HASHES" hatasi = yarim/bozuk inen wheel.
# Cozum: once cache'i temizle, sonra --no-cache-dir ile TAZE indir.
!pip cache purge
!pip install -q --no-cache-dir unsloth


In [ ]:
# ===== HÜCRE 3: İçe aktarmalar (imports) =====
from unsloth import FastLanguageModel   # hızlı yükleyici (base + LoRA)
import torch, re, time, gc              # motor + yardımcılar
from datasets import load_dataset       # veri getirici
from tqdm.auto import tqdm              # ilerleme çubuğu
import pandas as pd                     # sonuç tablosu
from huggingface_hub import login       # kimlik doğrulama
from getpass import getpass             # token'ı gizli sormak için
print("Kütüphaneler hazır. GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ===== HÜCRE 4: HuggingFace girişi =====
# Token'ı ekrana yazmadan sorar (kod içine token GÖMMÜYORUZ - güvenlik).
login(getpass("HF token'ını yapıştır ve Enter: "))
print("Giris basarili.")


In [ ]:
# ===== HÜCRE 5: Veriyi yükle ve TANI (önce veriye bak!) =====
# Bu hücrenin ÇIKTISINI bana gönder: şık sayısı (4 mü 5 mi) ve cevap index aralığını
# birlikte doğrulayacağız. Prompt ve puanlama buna göre kesinleşecek.
ds = load_dataset("alibayram/turkish_mmlu", split="mmlu")
print("Soru sayisi:", len(ds))
print("Sutunlar:", ds.column_names)
print("\n--- Ilk ornek ---")
ornek = ds[0]
for k, v in ornek.items():
    print(f"{k}: {str(v)[:200]}")
print("\ncevap tipi:", type(ornek["cevap"]).__name__,
      "| gorulen cevap degerleri:", sorted(set(ds["cevap"])))
print("secenekler tipi:", type(ornek["secenekler"]).__name__,
      "| bu ornekte sik sayisi:", len(ornek["secenekler"]))


In [ ]:
# ===== HÜCRE 6: Soru -> Prompt (istem) donusturucu =====
# Sik sayisina gore ESNEK (4 -> A-D, 5 -> A-E). Modelden SADECE harf istiyoruz.
HARFLER = "ABCDE"

def prompt_yap(ornek):
    secenekler = "\n".join(
        f"{HARFLER[i]}) {metin}" for i, metin in enumerate(ornek["secenekler"])
    )
    return (
        "Asagidaki coktan secmeli soruyu yanitla. "
        "Sadece dogru sikkin harfini yaz (ornek: B). Baska hicbir sey yazma.\n\n"
        f"Soru: {ornek['soru']}\n{secenekler}\n\nCevap:"
    )

# Gorsel kontrol: ilk sorunun prompt'u nasil gorunuyor?
print(prompt_yap(ds[0]))


In [ ]:
# ===== HÜCRE 7: Cevap ayristirici (answer parsing) =====
# Modelin serbest metninden harfi ceker. Once "tek basina duran harf", sonra yedek.
def harf_bul(metin):
    metin = metin.strip().upper()
    m = re.search(r"\b([A-E])\b", metin)   # ' B ' gibi tek basina duran harf
    if m:
        return m.group(1)
    m = re.search(r"[A-E]", metin)          # yedek: ilk A-E harfi
    return m.group() if m else None

# Hizli test
for t in ["B", "Cevap: C", "D) dogru", "bilmiyorum"]:
    print(repr(t), "->", harf_bul(t))


In [ ]:
# ===== HÜCRE 8: Degerlendirme fonksiyonu (yukle + batch + puanla + belleği boşalt) =====
def degerlendir(model_adi, soru_sayisi=100, batch=8, load_in_4bit=True,
                karistir=True, seed=42):
    # 1) MODELI YUKLE: diskteki sayilari GPU bellegine al
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_adi, max_seq_length=2048, load_in_4bit=load_in_4bit,
    )
    FastLanguageModel.for_inference(model)          # "sinav modu": sadece uret, ogrenme yok
    tokenizer.padding_side = "left"                 # batch uretim icin sola dolgu
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Temsili olsun diye ONCE karistir. Sabit seed => iki model AYNI sorulari gorur (adil).
    kaynak = ds.shuffle(seed=seed) if karistir else ds
    altkume = kaynak.select(range(min(soru_sayisi, len(ds))))
    dogru, kayitlar = 0, []
    t0 = time.time()

    for i in tqdm(range(0, len(altkume), batch), desc=model_adi.split("/")[-1]):
        grup = altkume.select(range(i, min(i + batch, len(altkume))))
        # Prompt -> Qwen3 chat sablonu (dusunme modu KAPALI)
        metinler = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt_yap(o)}],
                tokenize=False, add_generation_prompt=True, enable_thinking=False,
            )
            for o in grup
        ]
        enc = tokenizer(metinler, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            cikti = model.generate(
                **enc, max_new_tokens=8, do_sample=False,   # greedy = tekrar uretilebilir
                pad_token_id=tokenizer.pad_token_id,
            )
        yeni = cikti[:, enc["input_ids"].shape[1]:]         # sadece YENI uretileni al
        cozulen = tokenizer.batch_decode(yeni, skip_special_tokens=True)

        for o, ham in zip(grup, cozulen):
            tahmin = harf_bul(ham)
            dogru_harf = HARFLER[o["cevap"]]                # sayi -> harf
            isabet = (tahmin == dogru_harf)
            dogru += int(isabet)
            kayitlar.append({
                "bolum": o["bolum"], "konu": o["konu"],       # konu-bazi kirilim icin
                "soru": o["soru"][:50], "ham_cikti": ham.strip(),
                "tahmin": tahmin, "dogru_cevap": dogru_harf, "isabet": isabet,
            })

    sure = time.time() - t0
    basari = dogru / len(altkume) * 100

    # 2) BELLEGI BOSALT: sonraki modele yer ac (iki 14B ayni anda sigmaz)
    del model, tokenizer
    gc.collect(); torch.cuda.empty_cache()

    ozet = {"model": model_adi, "soru_sayisi": len(altkume), "dogru": dogru,
            "basari_%": round(basari, 2), "sure_sn": round(sure, 1)}
    return ozet, kayitlar


In [ ]:
# ===== HÜCRE 9: Yalın Qwen3-14B (kontrol grubu) =====
# Tam resmi olcum: 6200 sorunun HEPSI. (soru_sayisi=6200 => karistir etkisiz, hepsi girer.)
base_ozet, base_kayit = degerlendir("unsloth/Qwen3-14B", soru_sayisi=6200)
base_ozet


In [ ]:
# ===== HÜCRE 10: Qwen3-14B + senin MEB LoRA'n (deney grubu) =====
# ONEMLI: bu koşuyu Hücre 9 ile AYNI oturumda/GPU'da yap (runtime degistirme).
lora_ozet, lora_kayit = degerlendir("namruni/meb-ogretmen-qwen3-14b-lora", soru_sayisi=6200)
lora_ozet


In [ ]:
# ===== HÜCRE 11: Karsilastirma tablosu (mini-leaderboard) =====
tablo = pd.DataFrame([base_ozet, lora_ozet])
tablo.insert(0, "etiket", ["Yalin Qwen3-14B (kontrol)", "Qwen3-14B + MEB LoRA (deney)"])
tablo


In [ ]:
# ===== HÜCRE 12: Ayristirma denetimi (model ne dedi, biz ne anladik?) =====
# Endise ettigin "sik belirleme" adimini GOZLE denetle. Ozellikle isabet==False satirlara bak:
# gercekten yanlis mi bildi, yoksa biz mi yanlis ayristirdik?
pd.DataFrame(base_kayit).head(20)


In [ ]:
# ===== HÜCRE 13: Konu (bolum) bazinda kirilim - fine-tune NEREDE ise yaradi? =====
# base ve lora kayitlarini bolume gore grupla, her bolumde dogrulugu kiyasla.
b = pd.DataFrame(base_kayit)
l = pd.DataFrame(lora_kayit)

kirilim = pd.DataFrame({
    "base_%": b.groupby("bolum")["isabet"].mean().mul(100).round(1),
    "lora_%": l.groupby("bolum")["isabet"].mean().mul(100).round(1),
    "soru":   b.groupby("bolum").size(),
})
kirilim["fark"] = (kirilim["lora_%"] - kirilim["base_%"]).round(1)   # + => LoRA iyilestirdi
kirilim = kirilim.sort_values("fark", ascending=False)

print(">>> LoRA'nin EN COK IYILESTIRDIGI 10 bolum:")
print(kirilim.head(10).to_string())
print("\n>>> LoRA'nin EN COK KOTULESTIRDIGI 10 bolum:")
print(kirilim.tail(10).to_string())
print(f"\nGenel: base %{b['isabet'].mean()*100:.1f} | lora %{l['isabet'].mean()*100:.1f}")
kirilim
